

```
vcbs_report_ingestion.ipynb          miniRAG_1.ipynb
─────────────────────────            ───────────────
Fetch API                           Load vcbs_reports.json
Filter + dedup                       Load vcbs_embeddings.npy (or embed here)
Clean + normalize                    embed_query()
Save vcbs_reports.json        →      cosine similarity
                                     topK
                                     LLM answer
                                     Gradio UI
                                     
                  ```



In [ ]:
import numpy as np
import pandas as pd
import json
import os
import re
from google.colab import drive, userdata

In [ ]:
drive.mount('/content/drive', force_remount=True)

BASE_DIR = "/content/drive/MyDrive/RAG_Project"

with open(f"{BASE_DIR}/vcbs_reports.json", "r", encoding="utf-8") as f:
    vcbs_reports = json.load(f)

doc_ids = [str(r["id"]) for r in vcbs_reports]
doc_texts = [f"{r['name']}. {r['description']}" for r in vcbs_reports]

print(f"Loaded {len(doc_ids)} VCBS reports")

Mounted at /content/drive
Loaded 1000 VCBS reports


In [ ]:
# Gemini
import os
from google.colab import userdata
import google.genai as genai

# Securely fetch the key from Colab Secrets
api_key = userdata.get('gemini-key')

# Initialize the client (this replaces genai.configure)
client = genai.Client(api_key=api_key)

In [ ]:
# Openrouter
import time
from openai import OpenAI
import numpy as np

# Nemotron client (set up once)
open_key = userdata.get('openrouter-key')
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=open_key
)

LOAD VCBS REPORTS

In [ ]:
df = pd.DataFrame(vcbs_reports)
df["article_id"] = df["id"].astype(str)
df["title"] = df["name"]
print(f"DataFrame: {len(df)} rows")
print(df.columns.tolist())

DataFrame: 1000 rows
['id', 'name', 'stockSymbol', 'category', 'description', 'publishedAt', 'pdf_url', 'locale', 'article_id', 'title']


In [ ]:
# Prepare embedding texts
def prepare_text(report, max_chars=3000):
    text = f"{report['name']}. {report['description']}"
    return text[:max_chars] if len(text) > max_chars else text

In [ ]:
def prepare_text(report, max_chars=3000):
    text = f"{report.get('name', '')}. {report.get('description', '')}"
    return text[:max_chars] if len(text) > max_chars else text

In [ ]:
doc_ids = [str(r["id"]) for r in vcbs_reports]
doc_texts = [prepare_text(r) for r in vcbs_reports]
print(f"Prepared {len(doc_ids)} reports for embedding")

Prepared 1000 reports for embedding


In [ ]:
MODELS = ["gemini-embedding-001"]

used_model = None
CACHE_FILE = f"{BASE_DIR}/vcbs_embeddings.npy"
MODEL_CACHE = f"{BASE_DIR}/vcbs_model.txt"
def embed_with_fallback(texts, batch_size=100):
    global used_model

    for model_name in MODELS:
        for attempt in range(3):
            try:
                all_embeddings = []
                for i in range(0, len(texts), batch_size):
                    batch = texts[i : i + batch_size]
                    result = client.models.embed_content(
                        model=model_name,
                        contents=batch
                    )
                    all_embeddings.extend([e.values for e in result.embeddings])

                used_model = model_name
                print(f"Embedded {len(texts)} text(s) using {model_name}")
                return np.array(all_embeddings)

            except Exception as e:
                error_str = str(e)
                if "429" in error_str and attempt < 2:
                    print(f"Rate limited. Waiting 45s (attempt {attempt + 1}/3)...")
                    time.sleep(45)
                else:
                    print(f"{model_name} failed: {e}")
                    break

    print("Gemini failed. Falling back to Nemotron...")
    try:
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            print(f"  Embedding batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}...")
            response = openrouter_client.embeddings.create(
              model="nvidia/nemotron-3-embed-1b:free",
              input=batch,
              encoding_format="float"
            )
            all_embeddings.extend([e.embedding for e in response.data])

        used_model = "nemotron-3-embed-1b"
        print(f"Embedded {len(texts)} text(s) using Nemotron (OpenRouter)")
        return np.array(all_embeddings)

    except Exception as e:
        print(f"Nemotron failed: {e}")

    raise RuntimeError("All embedding models failed")


# Load from cache or embed fresh
if os.path.exists(CACHE_FILE) and os.path.exists(MODEL_CACHE):
    doc_embeddings = np.load(CACHE_FILE)
    with open(MODEL_CACHE, "r") as f:
        used_model = f.read().strip()
    print(f"Loaded cached embeddings: {doc_embeddings.shape} (model: {used_model})")
else:
    doc_embeddings = embed_with_fallback(doc_texts,20)
    np.save(CACHE_FILE, doc_embeddings)
    with open(MODEL_CACHE, "w") as f:
        f.write(used_model)
    print(f"Saved embeddings: {doc_embeddings.shape} (model: {used_model})")

Rate limited. Waiting 45s (attempt 1/3)...
Rate limited. Waiting 45s (attempt 2/3)...
gemini-embedding-001 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 74.425981ms.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerM

In [ ]:
user_response = input("Enter query: ")
# Tokenize query
parsed_query = user_response.lower().split(" ")

Enter query: what's the latest report on VCBS about?


In [ ]:
# Find query term from parsed_desc
# Store documents in a dictionary
def find_query_term(inverted_index, parsed_query):
  query_terms = {}

  for term in parsed_query:
    if term in inverted_index:
      query_terms[term] = inverted_index[term]

  return query_terms

In [ ]:
def tokenize(df):
    parsed_desc = {}

    for _, row in df.iterrows():
        article_id = str(row["article_id"])
        description = row["description"]

        if pd.notna(description):
            parsed_desc[article_id] = str(description).lower().split()
        else:
            parsed_desc[article_id] = []

    return parsed_desc

In [ ]:
def build_inverted_index(parsed_desc):
    inverted_index = {}

    for document_id, document in parsed_desc.items():
        unique_terms = set(document)

        for term in unique_terms:
            if term not in inverted_index:
                inverted_index[term] = set()

            inverted_index[term].add(document_id)

    return inverted_index

In [ ]:
inverted_index = build_inverted_index

In [ ]:
query_terms = find_query_term(inverted_index, parsed_query)
print(query_terms)

NameError: name 'inverted_index' is not defined

In [ ]:
# Embed query

def embed_query(text):
    if used_model == "nemotron-3-embed-1b":
        response = openrouter_client.embeddings.create(
            model="nvidia/nemotron-3-embed-1b:free",
            input=[text],
            encoding_format="float"
        )
        return np.array([response.data[0].embedding])

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[text]
    )
    return np.array([result.embeddings[0].values])

In [ ]:
# Check cosine similarity

from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(query_embedding, doc_embeddings)[0]

# Build a dict of {article_id: score} — same shape topK expects
embedding_scores = {}
for i in range(len(doc_ids)):
    embedding_scores[doc_ids[i]] = scores[i]

# Sort descending by score
sorted_embedding = dict(sorted(embedding_scores.items(), key=lambda x: x[1], reverse=True))

# Reuse your existing topK function
topK(sorted_embedding, df, 10)

[('68817',
  np.float64(0.5318485233753766),
  'Bull Market? Bear Market? Something Else? What Should Investors DO?',
  'Is this a new bear market, and if so, how can investors adapt? Is this still a bull market, and investors should just take a deep breath and buy? Or is it a completely DIFFERENT kind of market, one more akin to what we saw in the 80s and 90s rather than 2009-â\x80¦'),
 ('69166',
  np.float64(0.5283833051736526),
  'Bull Market? Bear Market? Something Else? What Should Investors DO?',
  'A new bear market?...How can investors adapt? Is this still a bull market, and should investors just take a deep breath and buy? Or a completely DIFFERENT kind of market?'),
 ('43793',
  np.float64(0.4548240732463333),
  'Go East For A Bull Market',
  'The big bright spot this year has been the Japanese market, where stocks look to have entered into what looks like a very powerful bull market.'),
 ('209515',
  np.float64(0.4481538579378688),
  'If there is value, buyers will find you 

In [ ]:
# Declare the 2 vectors
A = [2,3]
B = [5,1]

In [ ]:
print(A)

[2, 3]


In [ ]:
print(B)

[5, 1]


In [ ]:
# calculate dot product

# if A contains mor elements than B, then length = A; else length = B
if A > B:
  length = len(A) #length A
else:
  length = len(B) #length B

# Renamed 'sum' to 'dot_product_sum' to avoid shadowing built-in sum()
dot_product_sum = 0
for i in range(length):
  dot_product_sum = dot_product_sum + A[i] * B[i]

In [ ]:
# print summ
print(dot_product_sum)

13


In [ ]:
# calculate cosine similarity
# cos similarity = dot product /  ||A|| * ||B||

lengthA = math.sqrt(A[0]**2 + A[1]**2) # ** is exponent
lengthB = math.sqrt(B[0]**2 + B[1]**2)


cosAB = dot_product_sum / (lengthA*lengthB)

print(cosAB)

0.7071067811865476


In [ ]:
# calculate euclidean distance
euclidean = math.sqrt((B[0] - A[0])**2 + (B[1] - A[1])**2)
print(euclidean)

3.605551275463989


Testing TF Concept

In [ ]:
doc1 = "test test test apple watermelon mango"
doc2 = "it's very hot today"
doc3 = "i drank apple and watermelon juice today because it was hot"

# cut into substrings first aka tokenization
token1 = doc1.split(" ")
token2 = doc2.split(" ")
token3 = doc3.split(" ")


In [ ]:
# posting list token1
# This gives "test": ["doc1", "doc1", "doc1"] when test appears three times.
# That mixes posting list (which should be doc IDs) with term frequency
# (which you compute later anyway).

terms = {}
for word in token1:
    if word not in terms:
       terms[word] = []
    terms[word].append('doc1')

print(terms)


{'test': ['doc1', 'doc1', 'doc1'], 'apple': ['doc1'], 'watermelon': ['doc1'], 'mango': ['doc1']}


In [ ]:
# posting list token2
for word in token2:
    if word not in terms:
       terms[word] = []
    terms[word].append('doc2')

print(terms)

{'test': ['doc1', 'doc1', 'doc1'], 'apple': ['doc1'], 'watermelon': ['doc1'], 'mango': ['doc1'], "it's": ['doc2'], 'very': ['doc2'], 'hot': ['doc2'], 'today': ['doc2']}


In [ ]:
# posting list token3
for word in token3:
    if word not in terms:
       terms[word] = []
    terms[word].append('doc3')

print(terms)

{'test': ['doc1', 'doc1', 'doc1'], 'apple': ['doc1', 'doc3'], 'watermelon': ['doc1', 'doc3'], 'mango': ['doc1'], "it's": ['doc2'], 'very': ['doc2'], 'hot': ['doc2', 'doc3'], 'today': ['doc2', 'doc3'], 'i': ['doc3'], 'drank': ['doc3'], 'and': ['doc3'], 'juice': ['doc3'], 'because': ['doc3'], 'it': ['doc3'], 'was': ['doc3']}


In [ ]:
# Calculate TF for each term for each doc
tf_count = {}
for word in terms:
    tf_count[word] = {}
    tf_count[word]['doc1'] = token1.count(word)
    tf_count[word]['doc2'] = token2.count(word)
    tf_count[word]['doc3'] = token3.count(word)

print(tf_count)

{'test': {'doc1': 3, 'doc2': 0, 'doc3': 0}, 'apple': {'doc1': 1, 'doc2': 0, 'doc3': 1}, 'watermelon': {'doc1': 1, 'doc2': 0, 'doc3': 1}, 'mango': {'doc1': 1, 'doc2': 0, 'doc3': 0}, "it's": {'doc1': 0, 'doc2': 1, 'doc3': 0}, 'very': {'doc1': 0, 'doc2': 1, 'doc3': 0}, 'hot': {'doc1': 0, 'doc2': 1, 'doc3': 1}, 'today': {'doc1': 0, 'doc2': 1, 'doc3': 1}, 'i': {'doc1': 0, 'doc2': 0, 'doc3': 1}, 'drank': {'doc1': 0, 'doc2': 0, 'doc3': 1}, 'and': {'doc1': 0, 'doc2': 0, 'doc3': 1}, 'juice': {'doc1': 0, 'doc2': 0, 'doc3': 1}, 'because': {'doc1': 0, 'doc2': 0, 'doc3': 1}, 'it': {'doc1': 0, 'doc2': 0, 'doc3': 1}, 'was': {'doc1': 0, 'doc2': 0, 'doc3': 1}}


In [ ]:
# Calculate IDF for each term
idf_count = {}
N = 3
for word in tf_count.keys():
  count = 0
  for doc in tf_count[word].keys():
        if tf_count[word][doc] > 0:
          count = count + 1
  idf_count[word] = math.log(N/count)

print(idf_count)

{'test': 1.0986122886681098, 'apple': 0.4054651081081644, 'watermelon': 0.4054651081081644, 'mango': 1.0986122886681098, "it's": 1.0986122886681098, 'very': 1.0986122886681098, 'hot': 0.4054651081081644, 'today': 0.4054651081081644, 'i': 1.0986122886681098, 'drank': 1.0986122886681098, 'and': 1.0986122886681098, 'juice': 1.0986122886681098, 'because': 1.0986122886681098, 'it': 1.0986122886681098, 'was': 1.0986122886681098}


In [ ]:
# Calculate TF-IDF per term per doc
tf_idf = {}
for word in tf_count.keys():
  tf_idf[word] = {}
  for doc in tf_count[word].keys():
    tf_idf[word][doc] = tf_count[word][doc] * idf_count[word]

print(tf_idf)

{'test': {'doc1': 3.295836866004329, 'doc2': 0.0, 'doc3': 0.0}, 'apple': {'doc1': 0.4054651081081644, 'doc2': 0.0, 'doc3': 0.4054651081081644}, 'watermelon': {'doc1': 0.4054651081081644, 'doc2': 0.0, 'doc3': 0.4054651081081644}, 'mango': {'doc1': 1.0986122886681098, 'doc2': 0.0, 'doc3': 0.0}, "it's": {'doc1': 0.0, 'doc2': 1.0986122886681098, 'doc3': 0.0}, 'very': {'doc1': 0.0, 'doc2': 1.0986122886681098, 'doc3': 0.0}, 'hot': {'doc1': 0.0, 'doc2': 0.4054651081081644, 'doc3': 0.4054651081081644}, 'today': {'doc1': 0.0, 'doc2': 0.4054651081081644, 'doc3': 0.4054651081081644}, 'i': {'doc1': 0.0, 'doc2': 0.0, 'doc3': 1.0986122886681098}, 'drank': {'doc1': 0.0, 'doc2': 0.0, 'doc3': 1.0986122886681098}, 'and': {'doc1': 0.0, 'doc2': 0.0, 'doc3': 1.0986122886681098}, 'juice': {'doc1': 0.0, 'doc2': 0.0, 'doc3': 1.0986122886681098}, 'because': {'doc1': 0.0, 'doc2': 0.0, 'doc3': 1.0986122886681098}, 'it': {'doc1': 0.0, 'doc2': 0.0, 'doc3': 1.0986122886681098}, 'was': {'doc1': 0.0, 'doc2': 0.0, 'do

In [ ]:
user_response = input("Enter query: ")

Enter query: test


In [ ]:
# Tokenize query
user_response.split(" ")

['test']

In [ ]:
# Renamed 'sum' to 'query_scores_sum' to avoid shadowing built-in sum() function
query_scores_sum = {}
for word in user_response.split():
  if word in tf_idf.keys():
    for doc in tf_idf[word].keys():
      if doc not in query_scores_sum.keys():
        query_scores_sum[doc] = 0
      query_scores_sum[doc] = query_scores_sum[doc] + tf_idf[word][doc]

print(query_scores_sum)

{'doc1': 3.295836866004329, 'doc2': 0.0, 'doc3': 0.0}


In [ ]:
# Sort descending using the updated variable name
sorted_doc = dict(sorted(query_scores_sum.items(), reverse=True))
print(sorted_doc)

{'doc3': 0.0, 'doc2': 0.0, 'doc1': 3.295836866004329}


CSV Clean Up TO BE USED LATER

In [ ]:
df = pd.read_csv('data.csv', on_bad_lines='skip', engine='python', encoding='latin1')

In [ ]:
df = df.drop(columns=['source_id', 'source_name', 'author', 'url', 'url_to_image', 'published_at', 'category','content','full_content'], errors='ignore')

In [ ]:
df["article_id"] = df["article_id"].astype(str)

# Drop duplicates to ensure the index is unique for the dictionary conversion
article_lookup = df.drop_duplicates(subset="article_id").set_index("article_id")[["title", "description"]].to_dict("index")

In [ ]:
import re

def is_english(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return False
    ascii_chars = len(re.findall(r'[a-zA-Z]', text))
    return ascii_chars / max(len(text), 1) > 0.5

df = df[df['description'].apply(is_english)].reset_index(drop=True)

Manipulate Dataset

In [ ]:
parsed_desc = tokenize(df)

In [ ]:
# TF Per Article Func -> basic ver first
# for each doc, count word
# for each word, tally
# probably same structure -> dictionary with doc, and then instead of list of words, list freq

def find_tf(desc):
  tf_per_doc = {}

  for document_id, document in desc.items():
      counts = Counter(document)
      total_words = len(document)

      if total_words == 0:
          tf_per_doc[document_id] = {}
          continue

      tf_per_doc[document_id] = {
          term: count / total_words
          for term, count in counts.items()
      }

  return tf_per_doc

In [ ]:
# IDF Per Word Per Doc
def find_idf(desc):
    document_frequency = Counter()

    for document in desc.values():
        unique_terms = set(document)

        for term in unique_terms:
            document_frequency[term] += 1

    total_documents = len(desc)

    idf = {
        term: math.log(
            (total_documents - document_count + 0.5)
            / (document_count + 0.5)
            + 1
        )        for term, document_count in document_frequency.items()
    }

    return idf

In [ ]:
def find_TFIDF(tf_per_doc, idf):
    tf_idf_by_doc = {}

    for document_id, document_tf in tf_per_doc.items():
        tf_idf_by_doc[document_id] = {}

        for term, tf_value in document_tf.items():
            tf_idf_by_doc[document_id][term] = (
                tf_value * idf[term]
            )

    return tf_idf_by_doc

In [ ]:
inverted_index = build_inverted_index(parsed_desc)
tf_per_doc = find_tf(parsed_desc)
idf = find_idf(parsed_desc)
parsed_desc = find_TFIDF(tf_per_doc, idf)

In [ ]:
# BM25 Steps:
# 1. Checks how many times each term appears.
# 2. Gives rarer terms more weight.
# 3. Reduces the benefit of repeated terms.
# 4. Adjusts for document length.
# 5. Adds the scores for all query terms.
# Use helper method for calculation
def countAppearance(parseddesc, parsedquery):
    appearance = {}

    for articleid, tokens in parseddesc.items():
        for term in parsedquery:
            count = tokens.count(term)

            if count > 0:
                if articleid not in appearance:
                    appearance[articleid] = {}
                appearance[articleid][term] = count

    return appearance

In [ ]:
def BM25(parsedquery, parseddesc, idf, k1=1.5, b=0.75):
    doclengths = {
        documentid: len(document)
        for documentid, document in parseddesc.items()
    }

    if not doclengths:
        return {}

    avgdl = sum(doclengths.values()) / len(doclengths)

    appearance = countAppearance(parseddesc, parsedquery)

    scores = {docid: 0.0 for docid in appearance}

    for docid, termcounts in appearance.items():
        doclen = doclengths.get(docid, 0)
        if doclen == 0:
            continue

        for term, tf in termcounts.items():
            if term not in idf:
                continue

            idft = idf[term]
            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * (doclen / avgdl))

            scores[docid] += idft * (numerator / denominator)

    return scores

In [ ]:
def topK(sorted_doc, df, top_k):
    article_lookup = (
        df.assign(article_id=df["article_id"].astype(str))
          .drop_duplicates(subset=["article_id"])
          .set_index("article_id")[["title", "description"]]
          .to_dict("index")
    )
    results = []
    for article_id, score in list(sorted_doc.items())[:top_k]:
        article = article_lookup.get(str(article_id), {})
        results.append((article_id, score, article.get("title", "N/A"), article.get("description", "N/A")))
    return results

In [ ]:
# Func Print Top K
def printTopK(results, label="BM25 score"):
    if not results:
        print("No results found.")
        return
    for rank, (article_id, score, title, desc) in enumerate(results, 1):
        print(f"Rank {rank}:")
        print(f"Article ID: {article_id}")
        print(f"Title: {title}")
        print(f"Description: {desc}")
        print(f"{label}: {score:.4f}")
        print()

In [ ]:
raw_tokens = tokenize(df)
# print(countAppearance(raw_tokens, parsed_query))

bm25scores = BM25(parsed_query, raw_tokens, idf)
sorteddoc = dict(sorted(bm25scores.items(), key=lambda x: x[1], reverse=True))
bm25topK = topK(sorteddoc, df, 5)
printTopK(bm25topK, "BM25 score")

Rank 1:
Article ID: 134121
Title: What If This Time is Different For the Stock Market?
Description: I appreciate Benâs long-term view of stock market corrections but what if this time is different? What are the stats when the Fed is actively offloading trillions of assets AND raising rates? What if this cycle is an anomaly and should be treated as such? Thiâ¦
BM25 score: 19.4242

Rank 2:
Article ID: 44248
Title: Inside An NFT Collectorâs Mind
Description: What motivates buyers to invest in NFTs during a bear market and how their approach to collecting has changed.
BM25 score: 17.6811

Rank 3:
Article ID: 33254
Title: 33 Memes to Keep Coherent Thoughts at Bay
Description: I don't know about you, but I often get fed up with thinking. It's so annoying that we need to spend so much of our day using our conscious mind. What should I have for breakfast? What signoff should I use for that email? What do I have time to do at the weekâ¦
BM25 score: 16.8965

Rank 4:
Article ID: 27257
Title

In [ ]:
# TBD
# lower/uppercase issues, punctuations, etc -> add handling for it.
# encoder in latin may be the issue also

# **EMBEDDING**


```
                      OFFLINE (once, before any query)
                     ─────────────────────────────────
each document's description
         ↓
   embedding model
         ↓
   one vector per doc  ──→  store in a matrix (100 × 384)


                     ONLINE (per query)
                     ──────────────────
query string
         ↓
   same embedding model
         ↓
   one query vector  (1 × 384)
         ↓
   cosine similarity against every doc vector
         ↓
   rank by similarity → top K
```



In [ ]:
# collect descs -> lists

doc_ids = []
doc_texts = []

for _, row in df.iterrows():
    doc_ids.append(str(row["article_id"]))

    desc = row["description"]
    if pd.notna(desc):
        doc_texts.append(str(desc))
    else:
        doc_texts.append("")

print(f"Collected {len(doc_texts)} descriptions")
print(f"Example: {doc_texts[0][:100]}...")

Collected 68682 descriptions
Example: UN Secretary-General Antonio Guterres urged the world Monday to "stop the madness" of climate change...


MODEL INTEGRATION

In [ ]:
prompt = f"""
You are a professional assistant. Examine the sorted,
embedded documents and articles. Answer the user query in a detailed way,
including citations and references to the documents. For references, use the
title of the documents.

Content: {topK(sorted_embedding, df, 20)}

Question: {user_response}

Answer:
"""

# In the new SDK, use client.models.generate_content
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)

Based on the provided documents, the current stock market presents a complex and mixed picture characterized by volatility, mixed buy and sell signals, and varying regional trends. 

### What the Current Stock Market is Like
* **Mixed Signals and Volatility:** The stock market is currently grasping for direction, flashing multiple conflicting buy and sell signals at the same time (**"The stock market is grasping for direction as multiple buy and sell signals flash at the same time"**). It is operating in an unstable economic climate influenced by geopolitical unrest (such as the Middle East conflict), concerns regarding the Federal Reserve, and soaring Treasury yields (**"Big Tech earnings can make or break this market"**). 
* **Uncertain Market Phases:** Investors are actively questioning the overall trend—wondering if they are looking at a new bear market, a continuation of a bull market, or an entirely different economic environment akin to the 1980s and 1990s (**"Bull Market? Bear 

# GRADIO IMPLEMENTATION

In [ ]:
import gradio as gr

def search_and_answer(query):
    try:
        # 1. Embed query
        q_embedding = embed_query(query)

        # 2. Cosine similarity
        scores = cosine_similarity(q_embedding, doc_embeddings).ravel()

        # 3. Top-K
        embedding_scores = {}
        for i in range(len(doc_ids)):
            embedding_scores[doc_ids[i]] = float(scores[i])
        sorted_scores = dict(sorted(embedding_scores.items(), key=lambda x: x[1], reverse=True))

        results = topK(sorted_scores, df, 5)

        # 4. Format results + build context
        context = ""
        result_text = ""
        for rank, (aid, score, title, desc) in enumerate(results, 1):
            result_text += f"**Rank {rank}** (score: {score:.4f})\n"
            result_text += f"**{title}**\n{desc}\n\n"
            context += f"[{rank}] {title}\n{desc}\n\n"

        # 5. Ask Gemini
        prompt = f"""
You are a professional assistant. Answer the user query using the provided documents.
Include citations using document titles.

Content: {context}

Question: {query}

Answer:
"""

        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt
        )

        return result_text, response.text

    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        return f"Error: {error_msg}", str(e)

# Gradio 6.x compatible UI
with gr.Blocks(title="Mini RAG") as demo:
    gr.Markdown("# Mini RAG Search")
    query_input = gr.Textbox(label="Your question", placeholder="What's new?")
    search_btn = gr.Button("Search")
    results_output = gr.Markdown(label="Retrieved Documents")
    answer_output = gr.Textbox(label="RAG Answer", lines=20)

    search_btn.click(
        fn=search_and_answer,
        inputs=query_input,
        outputs=[results_output, answer_output]
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ea0ccea0756b11f772.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# `VCBS API IMPLEMENTATION`